## Selenium을 활용한 고양이 이미지 크롤링 🐱😺😸😹😻😼😽🙀😿😾

### 1. Selenium 라이브러리 설치 
파이썬으로 웹 브라우저를 원격 조종하기 위해 `selenium` 패키지를 설치합니다. 
(이미 설치되어 있다면 이 셀은 건너뛰어도 됩니다.)

In [ ]:
!pip install selenium

### 2. 필수 모듈 불러오기 
크롤링에 필요한 도구들을 가져옵니다. 
- `webdriver`: 실제 브라우저를 띄우고 조종하는 핵심 엔진
- `Keys`, `By`: 키보드 입력(엔터키 등)과 요소 탐색 기준(CSS, XPATH) 지정
- `time`: 페이지 로딩을 기다리기 위한 일시 정지용

In [13]:
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
import time

### 3. 브라우저 열고 네이버에 "고양이" 검색하기 
크롬 창을 열고 네이버 메인으로 이동한 뒤, 검색창을 찾아 "고양이"를 입력하고 엔터(RETURN)키를 칩니다.

In [14]:
driver = webdriver.Chrome()
driver.get("https://www.naver.com/")
time.sleep(2)

# 검색창 찾기 및 검색어 입력
search = driver.find_element(By.CSS_SELECTOR,".search_input")
search.send_keys("고양이")
search.send_keys(Keys.ENTER)
time.sleep(2)

### 4. 이미지 탭으로 이동하기 
통합검색 결과에서 '이미지' 탭을 클릭하여 사진 모아보기 화면으로 넘어갑니다. 
(참고: XPATH는 HTML 구조가 복잡할 때 절대 경로로 요소를 찾는 방법입니다.)

In [15]:
# 이미지 탭 클릭 (XPATH 예시)
image_tab = driver.find_element(By.CSS_SELECTOR,".tab") 
image_tab.click()

### 5. (맛보기) 첫 번째 이미지 하나만 클릭해보기 
반복문으로 수십 장을 저장하기 전에, 첫 번째 사진 하나만 클릭해서 원본 이미지 주소가 잘 뽑히는지 테스트해 봅니다.
(`find_elements`가 아니라 단수형인 `find_element`를 사용합니다.)

In [17]:
image = driver.find_element(By.CSS_SELECTOR,"._fe_image_tab_content_thumbnail_image")
image.click()

### 6. 무한 스크롤로 숨겨진 이미지 모두 불러오기 (해당 셀 실행 시 고양이 가득...)
보통 이미지 검색 결과는 스크롤을 내려야 추가로 로딩됩니다. 
자바스크립트 명령어를 사용해 브라우저 스크롤을 맨 아래로 계속 내리면서, 더 이상 새로운 이미지가 안 뜰 때까지 대기합니다.

In [ ]:
# 페이지의 끝까지 스크롤
last_height = driver.execute_script("return document.body.scrollHeight")

while True:
    # 페이지 끝까지 스크롤
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    # 로딩 대기
    time.sleep(2)
    # 새로운 높이 계산
    new_height = driver.execute_script("return document.body.scrollHeight")
    # 더 이상 스크롤이 내려가지 않으면(끝에 도달하면) 반복문 탈출
    if new_height == last_height:
        break
    last_height = new_height


### 7. 본격적인 이미지 수집 및 저장 
화면에 뜬 썸네일 이미지들을 하나씩 클릭하고, 확대된 원본 이미지의 주소(`src`)를 빼내어 우리 폴더에 `.jpg` 파일로 저장합니다.
수집이 모두 끝나면 브라우저를 안전하게 닫아줍니다.

In [ ]:
import os
import urllib.request

folder = "images"

if not os.path.exists(folder):
    os.makedirs(folder)

images = driver.find_elements(By.CSS_SELECTOR,"._fe_image_tab_content_thumbnail_image")
count = 1
for image in images:
    try:
        image.click()
        time.sleep(2)
        imgUrl = driver.find_element(By.CSS_SELECTOR,"._fe_image_viewer_image_fallback_target._fe_image_viewer_main_image").get_attribute("src")
        
        filepath = os.path.join(folder, str(count) + ".jpg")
        urllib.request.urlretrieve(imgUrl, filepath)
        
        count = count + 1
    except:
        pass

driver.close()